# Traitement Gold pour le pôle Marketing

## Création de la session Spark et chargement des données Silver

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("olist-gold-marketing") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

df_orders = spark.read.parquet("../data/silver/orders/")
df_customers = spark.read.parquet("../data/silver/customers/")
df_items = spark.read.parquet("../data/silver/order_items/")
df_payments = spark.read.parquet("../data/silver/payments/")
df_reviews = spark.read.parquet("../data/silver/reviews/")
df_products = spark.read.parquet("../data/silver/products/")
df_sellers = spark.read.parquet("../data/silver/sellers/")
df_translation = spark.read.parquet("../data/silver/product_category_name_translation/")

## Création de la table Gold

In [ ]:
from pyspark.sql.functions import col

df_base = df_orders \
    .join(df_customers, "customer_id", "left") \
    .join(df_items, "order_id", "left") \
    .join(df_products, "product_id", "left") \
    .join(df_translation, "product_category_name", "left") \
    .join(df_payments, "order_id", "left") \
    .join(df_reviews, "order_id", "left") \
    .join(df_sellers, "seller_id", "left")

# Ne garder que les commandes livrées pour les analyses marketing
df_base = df_base.filter(col("order_status") == "delivered")

print("Lignes dans le DataFrame central :", df_base.count())
df_base.printSchema()

## Analyse des données pour le pôle Marketing
### Satisfaction Globale

La note moyenne de satisfaction permet d'évaluer la qualité générale du service. Une note élevée (>4) indique une bonne expérience client, tandis qu'une note basse (<3) suggère des problèmes majeurs à traiter en priorité.

**Points clés :**
- Le nombre d'avis collectés reflète le taux de participation des clients au feedback
- Cette métrique doit être suivie mensuellement pour détecter les tendances (amélioration/dégradation)
- Les baisses anormales doivent déclencher une investigation immédiate

In [ ]:
from pyspark.sql.functions import avg, count, round

df_gold_satisfaction = df_reviews \
    .join(df_orders, "order_id", "left") \
    .filter(col("order_status") == "delivered") \
    .agg(
        round(avg("review_score"), 2).alias("note_moyenne"),
        count("review_id").alias("nombre_avis"),
    )

df_gold_satisfaction.show()
df_gold_satisfaction.write.mode("overwrite").parquet("../data/gold/marketing/satisfaction_globale/")

### Top Catégories

Cette analyse identifie les produits moteurs du chiffre d'affaires et de la satisfaction client.

**Actions marketing recommandées :**
- **Top performers** : Augmenter la visibilité, optimiser les stocks, recruter plus de vendeurs
- **Catégories sous-performantes mais demandées** : Analyser les raisons des faibles notes (logistique, qualité produit) et corriger
- **Opportunités de cross-selling** : Promouvoir les top catégories pour attirer du trafic sur les autres produits
- **Stratégie tarifaire** : Ajuster les prix des catégories haute demande/haute satisfaction

In [ ]:
from pyspark.sql.functions import sum as _sum, count, round, col

df_gold_categories = df_base \
    .groupBy("product_category_name_english") \
    .agg(
        round(_sum("price"), 2).alias("chiffre_affaires"),
        count("order_id").alias("nombre_commandes"),
        round(avg("review_score"), 2).alias("note_moyenne")
    ) \
    .filter(col("product_category_name_english").isNotNull()) \
    .orderBy(col("chiffre_affaires").desc())

df_gold_categories.show(20)
df_gold_categories.write.mode("overwrite").parquet("../data/gold/marketing/top_categories/")

### Analyse Géographique

Cette analyse révèle les disparités régionales en termes de volume et de satisfaction.

**Insights géographiques :**
- **Régions à fort potentiel** : Celles avec volume élevé ET satisfaction élevée = croissance durable
- **Régions problématiques** : Volume élevé mais satisfaction basse = risque de churn - Investigation urgente
- **Régions sous-exploitées** : Faible volume malgré bonne satisfaction = opportunité marketing
- **Logistique** : Analyser les délais de livraison par région pour expliquer les écarts de satisfaction

**Recommandations :**
- Déploiement de campagnes localisées pour les régions sous-exploitées
- Partenariats logistiques pour améliorer les services dans les régions problématiques
- Analyse concurrentielle par région

In [ ]:
from pyspark.sql.functions import count, round, sum as _sum

df_gold_geo = df_base \
    .groupBy("customer_state") \
    .agg(
        count("order_id").alias("nombre_commandes"),
        round(_sum("price"), 2).alias("chiffre_affaires"),
        round(avg("review_score"), 2).alias("note_moyenne")
    ) \
    .orderBy(col("chiffre_affaires").desc())

df_gold_geo.show(30)
df_gold_geo.write.mode("overwrite").parquet("../data/gold/marketing/geo_clients/")

### Évolution Mensuelle

Cette tendance temporelle permet de comprendre la dynamique du business et les cycles saisonniers.

**Analyse des tendances :**
- **Croissance soutenue** : Stratégie efficace, continuer les investissements
- **Décroissance** : Revoir la stratégie marketing, analyser la concurrence, relancer les campagnes
- **Variations saisonnières** : Préparer les stocks et la logistique pour les pics de demande
- **Corrélation satisfaction-volume** : Si les deux baissent, problème produit/service. Si satisfaction baisse avec volume en hausse, problème de qualité due à la croissance

**Actions :**
- Planifier les campagnes promotionnelles avant les pics saisonniers
- Recruter du personnel et augmenter les capacités avant les périodes de forte demande
- Identifier les mois faibles et proposer des incitations pour les booster

In [ ]:
from pyspark.sql.functions import date_format, count, round, sum as _sum

df_gold_mensuel = df_base \
    .withColumn("mois", date_format("order_purchase_timestamp", "yyyy-MM")) \
    .groupBy("mois") \
    .agg(
        count("order_id").alias("nombre_commandes"),
        round(_sum("price"), 2).alias("chiffre_affaires"),
        round(avg("review_score"), 2).alias("note_moyenne")
    ) \
    .orderBy("mois")

df_gold_mensuel.show(24)
df_gold_mensuel.write.mode("overwrite").parquet("../data/gold/marketing/evolution_mensuelle/")

### Fidélité Clients

Distribution du nombre de commandes par client : indicateur clé de la rétention et la LTV (Lifetime Value).

**Analyse de fidélité :**
- **Un achat** : Clients à risque de churn, taux d'acquisition inefficace - Besoin de relance urgente
- **2-3 achats** : Clients en transition, opportunité de fidélisation avec des offres loyauté
- **4+ achats** : Clients fidèles, haute LTV - À protéger, solliciter pour avis et témoignages
- **Distribution concentrée sur 1 achat** : Problème majeur de fidélisation

**Stratégies :**
- Programme de fidélité pour convertir les clients one-shot en clients réguliers
- Campagnes de réactivation 30 jours après le premier achat
- VIP program pour les clients 5+ achats (remises, livraison gratuite, early access)

In [ ]:
from pyspark.sql.functions import count, round

df_gold_fidelite = df_orders \
    .filter(col("order_status") == "delivered") \
    .groupBy("customer_id") \
    .agg(count("order_id").alias("nombre_commandes")) \
    .groupBy("nombre_commandes") \
    .agg(count("customer_id").alias("nombre_clients")) \
    .orderBy("nombre_commandes")

df_gold_fidelite.show()
df_gold_fidelite.write.mode("overwrite").parquet("../data/gold/marketing/fidelite_clients/")

### Acquisition Mensuelle

Nombre de nouveaux clients acquis chaque mois = indicateur de la performance des campagnes marketing.

**Analyse d'acquisition :**
- **Croissance progressive** : Stratégie marketing efficace, budgets marketing bien investis
- **Pics d'acquisition** : Identifier les campagnes/périodes qui les ont générés pour réplique
- **Creux d'acquisition** : Analyser les raisons (baisse de budget, concurrence, saisonnalité)
- **Corrélation acquisition/commandes** : Valider que les nouveaux clients restent actifs (ROI marketing)

**Métriques dérivées à calculer :**
- Coût d'acquisition par canal
- Taux de rétention 1er/2e achat des cohortes mensuelles
- LTV des clients acquis par mois/canal

**Recommandations :**
- Optimiser le budget marketing vers les canaux/périodes à meilleure conversion
- Planifier les campagnes pour lisser les creux saisonniers

In [ ]:
from pyspark.sql.functions import date_format, min as _min, count

# Première commande de chaque client = date d'acquisition
df_acquisition = df_orders \
    .filter(col("order_status") == "delivered") \
    .groupBy("customer_id") \
    .agg(_min("order_purchase_timestamp").alias("date_acquisition")) \
    .withColumn("mois_acquisition", date_format("date_acquisition", "yyyy-MM")) \
    .groupBy("mois_acquisition") \
    .agg(count("customer_id").alias("nouveaux_clients")) \
    .orderBy("mois_acquisition")

df_acquisition.show(24)
df_acquisition.write.mode("overwrite").parquet("../data/gold/marketing/acquisition_mensuelle/")

### NPS par Catégorie

Le Net Promoter Score mesure la propension des clients à recommander un produit (proxy). Indicateur clé de satisfaction et de fidélité.

**Lecture du NPS :**
- **NPS > 50** : Excellent, clients très satisfaits, fort potentiel de recommandation
- **NPS 0-50** : Bon, mais opportunités d'amélioration
- **NPS < 0** : Critique, risque de détracteurs qui parlent mal du produit
- **Nombre d'avis < 50** : Données insuffisamment significatives statistiquement

**Catégories enchanteurs (NPS haut) :**
- Augmenter le marketing et la visibilité
- Cas d'études à partager avec le reste de l'org
- Modèle à répliquer pour les autres catégories

**Catégories à problème (NPS bas) :**
- Investigation immédiate : qualité produit ? logistique ? prix ? 
- Focus sur l'expérience client avant toute promotion
- Considérer l'exclusion si pas d'amélioration rapide

In [ ]:
from pyspark.sql.functions import avg, count, round, col, when, sum as _sum

df_nps = df_base \
    .filter(col("review_score").isNotNull()) \
    .withColumn("promoteur",   when(col("review_score") == 5, 1).otherwise(0)) \
    .withColumn("detracteur",  when(col("review_score") <= 2, 1).otherwise(0)) \
    .groupBy("product_category_name_english") \
    .agg(
        count("review_id").alias("nb_avis"),
        round(avg("review_score"), 2).alias("note_moyenne"),
        round(
            (_sum("promoteur") - _sum("detracteur")) / count("review_id") * 100, 1
        ).alias("nps_score")  # proxy NPS : % promoteurs - % détracteurs
    ) \
    .filter(col("product_category_name_english").isNotNull()) \
    .filter(col("nb_avis") >= 50)  # seuil de significativité statistique

# Top catégories enchanteurs vs décevantes
print("=== Top catégories satisfaisantes ===")
df_nps.orderBy(col("nps_score").desc()).show(10)

print("=== Catégories à problème ===")
df_nps.orderBy(col("nps_score").asc()).show(10)

df_nps.write.mode("overwrite").parquet("../data/gold/marketing/nps_par_categorie/")

### Saisonnalité des Catégories

Identifier les patterns saisonniers par catégorie permet d'optimiser les stocks, budgets et planification.

**Patterns typiques :**
- **Électronique** : pics à Noël/Black Friday, creux en janvier
- **Mode** : pics saisonniers (été, hiver), soldes anticipées
- **Cadeaux/Jouets** : pics à Noël et anniversaires (concentrés)
- **Aliments/Basiques** : demande relativement stable

**Opportunités marketing :**
- **Anticiper les pics** : Augmenter les stocks, budgets marketing et partenaires 1-2 mois avant
- **Booster les creux** : Promotions, bundle deals, cross-selling avec catégories en pic
- **Planification des campagnes** : Aligner avec les comportements d'achat naturels
- **Recrutement logistique** : Préparer les effectifs pour les périodes de forte activité

In [ ]:
from pyspark.sql.functions import month, count, col

df_saisonnalite = df_base \
    .withColumn("mois", month("order_purchase_timestamp")) \
    .groupBy("product_category_name_english", "mois") \
    .agg(count("order_id").alias("nb_commandes")) \
    .filter(col("product_category_name_english").isNotNull()) \
    .orderBy("product_category_name_english", "mois")

# Focus sur les top 5 catégories
top5_categories = [
    row["product_category_name_english"]
    for row in df_base
        .groupBy("product_category_name_english")
        .count()
        .orderBy(col("count").desc())
        .limit(5)
        .collect()
]

df_saisonnalite \
    .filter(col("product_category_name_english").isin(top5_categories)) \
    .show(60)

df_saisonnalite.write.mode("overwrite").parquet("../data/gold/marketing/saisonnalite_categories/")

### Pénétration Géographique par Catégorie

Comprendre quelles catégories dominent dans chaque région permet des stratégies marketing ultra-ciblées.

**Insights géo-catégories :**
- **Part de marché élevée** : Catégorie leader régionale, forte demande locale
- **Part de marché faible** : Potentiel d'expansion ou déclin - à investiguer
- **Disparités régionales** : Certaines catégories sont régionales (ex: vêtements côtiers) vs nationales

**Stratégies par région :**
- **Régions dominées par une catégorie** : Partenariats avec leaders régionaux, sponsoring événementiel local
- **Régions diversifiées** : Approche généraliste, cross-selling opportuniste
- **Catégories sous-pénétrées** : Campagnes ciblées pour tester la demande locale

**Recommandations :**
- Créer des catalogues personnalisés par région (produits en vedette régionalisée)
- Adapter les campagnes emails/SMS par région et catégorie dominante
- Analyser pourquoi certaines catégories ne percent pas dans certaines régions

In [ ]:
from pyspark.sql.functions import count, col, round, sum as _sum

# Commandes totales par état
df_total_par_etat = df_base \
    .groupBy("customer_state") \
    .agg(count("order_id").alias("total_commandes_etat"))

# Commandes par état ET par catégorie
df_geo_categorie = df_base \
    .filter(col("product_category_name_english").isNotNull()) \
    .groupBy("customer_state", "product_category_name_english") \
    .agg(count("order_id").alias("nb_commandes")) \
    .join(df_total_par_etat, "customer_state") \
    .withColumn(
        "part_de_marche_pct",
        round(col("nb_commandes") / col("total_commandes_etat") * 100, 2)
    ) \
    .orderBy("customer_state", col("nb_commandes").desc())

df_geo_categorie.show(20)
df_geo_categorie.write.mode("overwrite").parquet("../data/gold/marketing/penetration_geo_categorie/")

### Affinité Produit (Market Basket Analysis)

Identifier les paires de catégories achetées ensemble permet d'optimiser le revenue par commande.

**Paires à haut potentiel :**
- **Fréquence élevée** : Combinaisons naturelles (ex: lit + draps, caméra + lentilles)
- **Opportunités de cross-sell** : Les clients achètent X, ils devraient acheter Y
- **Bundle opportuns** : Créer des packs de catégories complémentaires pour boost AOV

**Stratégies de cross-sell :**
1. **Sur le site** : Recommandations produits lors de la consultation/panier
2. **Campagnes email** : "Les clients qui ont acheté X achètent aussi Y"
3. **Packaging** : Insérer des coupons de réduction pour produits complémentaires
4. **Offres combo** : Réduction si achat de 2 catégories ensemble

**Gains potentiels :**
- Augmentation du panier moyen (AOV) sans hausse du trafic
- Réduction de la friction entre catégories concurrentes
- Amélioration de la satisfaction (clients trouvent tout en un lieu)

In [ ]:
from pyspark.sql.functions import col, count

# Catégories par commande
df_cat_par_commande = df_items \
    .join(df_products, "product_id") \
    .join(df_translation, "product_category_name") \
    .select("order_id", "product_category_name_english") \
    .distinct()

# Auto-jointure pour trouver les paires
df_affinite = df_cat_par_commande.alias("a") \
    .join(
        df_cat_par_commande.alias("b"),
        on=(col("a.order_id") == col("b.order_id")) &
           (col("a.product_category_name_english") < col("b.product_category_name_english"))
    ) \
    .groupBy(
        col("a.product_category_name_english").alias("categorie_1"),
        col("b.product_category_name_english").alias("categorie_2")
    ) \
    .agg(count("a.order_id").alias("achetes_ensemble")) \
    .orderBy(col("achetes_ensemble").desc())

print("=== Top paires de catégories achetées ensemble ===")
df_affinite.show(20)
df_affinite.write.mode("overwrite").parquet("../data/gold/marketing/affinite_produit/")